k겹 교차 검증이란 데이터셋을 여러 개로 나누어 하나씩 테스트셋으로 사용하고 나머지를 모두 합해서 학습셋으로 사용하는 방법이다. 이를 활용함으로서 데이터의 100%를 학습셋으로 사용할 수 있고, 또 동시에 테스트셋으로 사용할 수 있다.

데이터를 원하는 수만큼 나누어 각각 학습셋과 테스트셋으로 사용되게 하는 함수는 사이킷런 라이브러리의 KFold() 함수이다.

실습| 초음파 광물 예측하기: k겹 교차 검증

In [1]:
from re import VERBOSE
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score

import pandas as pd

# 깃허브에 준비된 데이터를 가져온다.
!git clone https://github.com/taehojo/data.git

# 광물 데이터를 불러온다.
df = pd.read_csv('./data/sonar3.csv', header=None)

# 음파 관련 속성을 X로, 광물의 종류를 y로 저장한다.
X = df.iloc[:, 0:60]
y = df.iloc[:, 60]

# 몇 겹으로 나눌 것인지 정한다.
k = 5

# KFold 함수를 불러온다. 분할하기 전에 샘플이 치우치지 앟도록 섞어준다.
kfold = KFold(n_splits=k, shuffle=True)

# 정확도가 채워질 빈 리스트를 준비한다.
acc_score = []
accuracy_sc = []
def model_fn():
    model = Sequential()
    model.add(Dense(24, input_dim=60, activation='relu'))
    model.add(Dense(10, activation='relu'))
    model.add(Dense(1, activation='sigmoid'))
    return model

# k겹 교차 검증을 이용해 k번의 학습을 실행한다.
# for 문에 의해 k번 반복한다.
# split()에 의해 k개의 학습셋, 테스트셋으로 분리된다.
for train_index, test_index in kfold.split(X):
    X_train, X_test = X.iloc[train_index, :], X.iloc[test_index, :]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model = model_fn()
    model.compile(loss='binary_crossentropy', optimizer='adam',
                  metrics=['accuracy'])
    history = model.fit(X_train, y_train, epochs=200, batch_size=10,
                        verbose=0)  # verbose=0 옵션을 주어 학습 과정의 출력을 생략

    accuracy = model.evaluate(X_test, y_test)[1]    # 정확도를 구한다.
    acc_score.append(accuracy)                      # 정확도 리스트에 저장한다.


# k번 실시된 정확도의 평균을 구한다.
avg_acc_score = sum(acc_score) / k

# 결과를 출력한다.
print('정확도:', acc_score)
print('정확도 평균:', avg_acc_score)

fatal: destination path 'data' already exists and is not an empty directory.
2/2 [==============================] - 0s 8ms/step - loss: 0.5195 - accuracy: 0.8049


2/2 [==============================] - 0s 11ms/step - loss: 0.5687 - accuracy: 0.8780
정확도: [0.7857142686843872, 0.9047619104385376, 0.738095223903656, 0.8048780560493469, 0.8780487775802612]
정확도 평균: 0.8222996473312378
